In [2]:
from dataclasses import dataclass
from typing import List, Dict

In [3]:
@dataclass
class ChoiceDecision:
  question: str
  options: List[str]

@dataclass
class BooleanDecision:
  question: str

In [4]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

In [12]:
class ZeroTokenDecisionEngine(nn.Module):
  def __init__(self, model_name: str = 'google/electra-base-discriminator'):
    super().__init__()
    self.tokenizer = AutoTokenizer.from_pretrained(model_name)
    self.encoder = AutoModel.from_pretrained(model_name)
    hidden_dim = self.encoder.config.hidden_size

    # binary calibration head(returns logit for P(true))
    self.binary_head = nn.Sequential(
      nn.Linear(hidden_dim, hidden_dim // 2),
      nn.GELU(),
      nn.Dropout(0.1),
      nn.Linear(hidden_dim // 2, 1)
    )

    # projection head for dynamic zero-shot semantic matching
    self.similarity_proj = nn.Linear(hidden_dim, hidden_dim)

  def _get_context_embedding(self, text: str) -> torch.Tensor:
    """extract the pooled representation of the state/question in one forward pass"""
    encoded = self.tokenizer(
        text,
        return_tensors= "pt",
        truncation = True,
        max_length = 512,
        padding = True
    )
    outputs = self.encoder(**encoded)
    # use [CLS] token representation
    return outputs.last_hidden_state[:, 0, :]

  def evaluate_boolean(self, context: str, question: str) -> float:
    """
    Evaluates a binary statement in a single forward pass without emitting tokens.
    eg: "does this message request account deletion?" -> P(True)
    """

    input_text = f"Context: {context} | Question: {question}"
    h = self._get_context_embedding(input_text)
    logit = self.binary_head(h)
    prob = torch.sigmoid(logit).item()
    return round(prob, 4)

  def evaluate_choice(self, context: str, decision: ChoiceDecision) -> Dict[str, float]:
    """
    calculates exact softmax probabilities over all options simultaneously
    -> no autoregression, no json generation
    """
    # 1. embed context + question
    query_text = f"Context: {context} | Task: {decision.question}"
    query_emb = self.similarity_proj(self._get_context_embedding(query_text))

    # 2. embed all candidate choices in a single batched forward pass
    candidate_encoded = self.tokenizer(
        decision.options,
        return_tensors= "pt",
        truncation = True,
        # max_length = 512,
        padding = True
    )

    with torch.no_grad():
      cand_outputs = self.encoder(**candidate_encoded)
      cand_embs = self.similarity_proj(cand_outputs.last_hidden_state[:, 0, :])

    # 3. compute dot-product logits between query and all candidate targets
    logits = torch.matmul(query_emb, cand_embs.T).squeeze(0)

    # 4. normalize to valid probability distribution
    probs = torch.softmax(logits, dim=-1).tolist()

    return {opt: round(prob, 4) for opt, prob in zip(decision.options, probs)}

In [13]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

class ZeroShotDecisionEngine:
    def __init__(self, model_name: str = "cross-encoder/nli-deberta-v3-base"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.eval()

    def evaluate_boolean(self, context: str, question: str) -> float:
        premise = context
        hypothesis = f"This statement is true: {question}."

        inputs = self.tokenizer(
            premise, hypothesis,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            logits = self.model(**inputs).logits
            contradiction_logit = logits[0, 0]
            entailment_logit = logits[0, 2]
            probs = torch.softmax(torch.stack([contradiction_logit, entailment_logit]), dim=-1)

        return round(probs[1].item(), 4)

    def evaluate_choice(self, context: str, options: list[str]) -> dict[str, float]:
        premise = context
        hypotheses = [f"This requires the action: {opt.replace('_', ' ')}." for opt in options]

        pairs = [[premise, hyp] for hyp in hypotheses]
        inputs = self.tokenizer(pairs, padding=True, truncation=True, return_tensors="pt")

        with torch.no_grad():
            logits = self.model(**inputs).logits
            entailment_logits = logits[:, 2]
            probs = torch.softmax(entailment_logits, dim=-1).tolist()

        return {opt: round(p, 4) for opt, p in zip(options, probs)}


# instantiate the new engine
engine = ZeroShotDecisionEngine()

user_query = "Can you refund my subscription for last month? I was charged twice."
current_state = f"User message: '{user_query}'. User status: Active paid tier."

# boolean check
is_refund_needed = engine.evaluate_boolean(
    context=current_state,
    question="The user is requesting financial compensation or a refund"
)
print(f"Policy Check - Needs Refund Handling: {is_refund_needed * 100:.1f}%")

# tool routing
options = [
    "search_documentation",
    "fetch_billing_ledger",
    "reset_password_workflow",
    "handoff_to_human_agent"
]

probabilities = engine.evaluate_choice(current_state, options)
print("\nTool Routing Probabilities:")
for tool, p in sorted(probabilities.items(), key=lambda x: x[1], reverse=True):
    print(f"  - {tool:25s}: {p*100:5.1f}%")

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.35k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Policy Check - Needs Refund Handling: 96.7%

Tool Routing Probabilities:
  - handoff_to_human_agent   :  44.9%
  - fetch_billing_ledger     :  24.7%
  - reset_password_workflow  :  17.7%
  - search_documentation     :  12.8%
